# 🍱 Food-101 MobileNetV2 분류기 (Kaggle 환경)

| 항목 | 내용 |
|------|------|
| 데이터셋 | Food-101 (101 클래스, 75,750 train / 25,250 test) |
| 모델 | MobileNetV2 (ImageNet pretrained) + Custom Head |
| Head | GAP → Dense(256, relu) → Dropout(0.5) → Dense(101, softmax) |
| 저장 경로 | `/kaggle/working/food_classifier.h5` |

## ⚙️ 실행 전 Kaggle 설정
1. **우측 패널 → Add Input → Search datasets → `food-101` 선택** (`dansbecker/food-101`)
2. **Session options → Accelerator → GPU T4 x2** (또는 GPU P100)
3. 셀 순서대로 실행

In [ ]:
# ── 1. 환경 확인 ───────────────────────────────────────────────────────
import os, pathlib, json
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SAVE_DIR  = '/kaggle/working/'   # 학습 결과 저장 위치
CACHE_DIR = '/kaggle/working/cache/'
os.makedirs(CACHE_DIR, exist_ok=True)

print(f'TensorFlow: {tf.__version__}')
print(f'GPU: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── 2. Food-101 데이터셋 로드 (다운로드 불필요 — Kaggle 입력 사용) ─────
# Add Input 에서 dansbecker/food-101 을 추가했어야 합니다
DATA_ROOT = pathlib.Path('/kaggle/input/food-101/food-101')
IMAGES_DIR = DATA_ROOT / 'images'
META_DIR   = DATA_ROOT / 'meta'

# ── 클래스 목록 로드
with open(META_DIR / 'classes.txt') as f:
    CLASS_NAMES = [line.strip() for line in f]
NUM_CLASSES  = len(CLASS_NAMES)
CLASS_TO_IDX = {name: i for i, name in enumerate(CLASS_NAMES)}
IDX_TO_CLASS = {i: name for i, name in enumerate(CLASS_NAMES)}

# ── 공식 train / test split 로드
def parse_split(txt_path):
    """각 줄: 'class_name/image_id' → (파일 경로, 레이블 인덱스) 리스트 반환"""
    paths, labels = [], []
    with open(txt_path) as f:
        for line in f:
            entry      = line.strip()              # e.g. 'apple_pie/1234'
            class_name = entry.split('/')[0]
            img_path   = str(IMAGES_DIR / entry) + '.jpg'
            paths.append(img_path)
            labels.append(CLASS_TO_IDX[class_name])
    return paths, labels

train_paths, train_labels = parse_split(META_DIR / 'train.txt')
val_paths,   val_labels   = parse_split(META_DIR / 'test.txt')

print(f'클래스 수:    {NUM_CLASSES}')
print(f'학습 샘플 수: {len(train_paths):,}')
print(f'검증 샘플 수: {len(val_paths):,}')
print(f'클래스 예시:  {CLASS_NAMES[:8]} ...')

In [ ]:
# ── 3. tf.data 파이프라인 구성 ─────────────────────────────────────────
IMG_SIZE   = 224
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

def load_and_preprocess(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.cast(tf.image.resize(img, [IMG_SIZE, IMG_SIZE]), tf.float32)
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, max_delta=30)
        img = tf.image.random_contrast(img, 0.8, 1.2)
        img = tf.image.random_saturation(img, 0.8, 1.2)
        img = tf.image.random_hue(img, 0.05)
        img = tf.clip_by_value(img, 0.0, 255.0)
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)  # [0,255] → [-1,1]
    return img, label

# 학습 파이프라인 (Augmentation + 디스크 캐시)
ds_train = (
    tf.data.Dataset.from_tensor_slices((train_paths, train_labels))
    .shuffle(len(train_paths), seed=42)
    .map(lambda p, l: load_and_preprocess(p, l, augment=True), num_parallel_calls=AUTOTUNE)
    .cache(CACHE_DIR + 'train')  # 디스크 캐시 (2회차 에폭부터 빠름)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

# 검증 파이프라인 (Augmentation 없음)
ds_val = (
    tf.data.Dataset.from_tensor_slices((val_paths, val_labels))
    .map(lambda p, l: load_and_preprocess(p, l, augment=False), num_parallel_calls=AUTOTUNE)
    .cache(CACHE_DIR + 'val')
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE)
)

print('파이프라인 구성 완료')

In [ ]:
# ── 4. 샘플 시각화 ────────────────────────────────────────────────────
sample_imgs, sample_labels = next(iter(ds_train))

fig, axes = plt.subplots(3, 6, figsize=(18, 9))
for i, ax in enumerate(axes.flatten()):
    if i >= len(sample_imgs): break
    img = np.clip((sample_imgs[i].numpy() + 1.0) / 2.0, 0, 1)
    ax.imshow(img)
    ax.set_title(IDX_TO_CLASS[int(sample_labels[i])], fontsize=7)
    ax.axis('off')
plt.suptitle('학습 데이터 샘플 (Data Augmentation 적용)', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5. MobileNetV2 + Custom Head 모델 ────────────────────────────────
def build_model(num_classes=NUM_CLASSES):
    base = tf.keras.applications.MobileNetV2(
        input_shape=(IMG_SIZE, IMG_SIZE, 3),
        include_top=False,
        weights='imagenet'
    )
    base.trainable = False  # Phase 1: 완전 동결

    inputs  = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base(inputs, training=False)
    x       = tf.keras.layers.GlobalAveragePooling2D()(x)
    x       = tf.keras.layers.Dense(256, activation='relu')(x)
    x       = tf.keras.layers.Dropout(0.5)(x)
    outputs = tf.keras.layers.Dense(num_classes, activation='softmax')(x)

    return tf.keras.Model(inputs, outputs, name='food_classifier'), base

model, base_model = build_model()
model.summary()

total     = model.count_params()
trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'\n전체 파라미터:      {total:,}')
print(f'학습 가능 파라미터: {trainable:,} ({trainable/total*100:.1f}%)')

In [ ]:
# ── 6. Phase 1: Custom Head 학습 (Base 동결) ──────────────────────────
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=4, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=2, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        SAVE_DIR + 'ckpt_phase1.keras',
        save_best_only=True, monitor='val_accuracy', verbose=0),
]

print('=== Phase 1: Custom Head 학습 (최대 15 epochs) ===')
history1 = model.fit(ds_train, validation_data=ds_val, epochs=15, callbacks=cb1)
print(f'\nPhase 1 최고 val_accuracy: {max(history1.history["val_accuracy"]):.4f}')

In [ ]:
# ── 7. Phase 2: Fine-tuning (마지막 30 레이어 언프리즈) ────────────────
FINETUNE_LAYERS = 30

base_model.trainable = True
for layer in base_model.layers[:-FINETUNE_LAYERS]:
    layer.trainable = False

trainable = sum(np.prod(v.shape) for v in model.trainable_variables)
print(f'Phase 2 학습 가능 파라미터: {trainable:,}')

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

cb2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=6, restore_best_weights=True, verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5, patience=3, verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        SAVE_DIR + 'ckpt_phase2.keras',
        save_best_only=True, monitor='val_accuracy', verbose=0),
]

print('=== Phase 2: Fine-tuning (최대 25 epochs) ===')
history2 = model.fit(
    ds_train, validation_data=ds_val, epochs=25,
    callbacks=cb2, initial_epoch=len(history1.history['accuracy'])
)
print(f'\nPhase 2 최고 val_accuracy: {max(history2.history["val_accuracy"]):.4f}')

In [ ]:
# ── 8. 학습 결과 시각화 ───────────────────────────────────────────────
acc   = history1.history['accuracy']     + history2.history['accuracy']
val   = history1.history['val_accuracy'] + history2.history['val_accuracy']
loss  = history1.history['loss']         + history2.history['loss']
vloss = history1.history['val_loss']     + history2.history['val_loss']
ep    = range(1, len(acc) + 1)
p2_start = len(history1.history['accuracy'])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
for ax, y, yv, lbl in [(ax1, acc, val, 'Accuracy'), (ax2, loss, vloss, 'Loss')]:
    ax.plot(ep, y,  label='Train')
    ax.plot(ep, yv, label='Val')
    ax.axvline(p2_start, color='gray', ls='--', label='Fine-tune 시작')
    ax.set_title(f'Food-101 MobileNetV2 — {lbl}')
    ax.set_xlabel('Epoch'); ax.set_ylabel(lbl)
    ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# ── 9. 최종 평가 ──────────────────────────────────────────────────────
from sklearn.metrics import classification_report

val_loss, val_acc = model.evaluate(ds_val, verbose=1)
print(f'\n최종 검증 정확도: {val_acc*100:.2f}%  |  손실: {val_loss:.4f}')

y_true, y_pred = [], []
for imgs, labels in ds_val:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print('\n전체 분류 리포트:')
print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))

In [ ]:
# ── 10. 모델 저장 ─────────────────────────────────────────────────────
MODEL_PATH     = SAVE_DIR + 'food_classifier.h5'
CLASS_IDX_PATH = SAVE_DIR + 'class_indices.json'

model.save(MODEL_PATH)
print(f'✅ 모델 저장: {MODEL_PATH}')

class_indices = {str(i): name for i, name in enumerate(CLASS_NAMES)}
with open(CLASS_IDX_PATH, 'w', encoding='utf-8') as f:
    json.dump(class_indices, f, ensure_ascii=False, indent=2)
print(f'✅ 클래스 인덱스 저장: {CLASS_IDX_PATH}')

print('\n' + '='*55)
print('📌 백엔드 연동 방법')
print('  1. Output 탭에서 두 파일 다운로드:')
print('       food_classifier.h5')
print('       class_indices.json')
print('  2. backend/app/ 폴더에 복사')
print('  3. pip install tensorflow Pillow numpy')
print('  4. 백엔드 서버 재시작')
print('  5. GET /api/health → ml_model_loaded: true 확인')
print('='*55)

In [ ]:
# ── 11. 추론 테스트 ───────────────────────────────────────────────────
test_imgs, test_labels = next(iter(ds_val))
preds = model.predict(test_imgs[:16], verbose=0)

fig, axes = plt.subplots(2, 8, figsize=(20, 6))
for i, ax in enumerate(axes.flatten()):
    img = np.clip((test_imgs[i].numpy() + 1.0) / 2.0, 0, 1)
    ax.imshow(img)
    true_cls = IDX_TO_CLASS[int(test_labels[i])]
    pred_cls = IDX_TO_CLASS[int(np.argmax(preds[i]))]
    conf     = float(np.max(preds[i]))
    color    = 'green' if true_cls == pred_cls else 'red'
    ax.set_title(f'{pred_cls}\n({conf:.0%})', color=color, fontsize=7)
    ax.axis('off')
plt.suptitle('추론 결과 (초록=정답, 빨강=오답)', fontsize=12)
plt.tight_layout()
plt.show()